In [1]:
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

def english_tokens(text):
    text = text.lower()
    return re.findall(r"[a-z]+", text)

def hindi_tokens(text):
    text = normalize_unicode(text)
    # Devanagari words; punctuation/numbers are excluded.
    return re.findall(r"[\u0900-\u097F]+", text)

def average_vectors(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

print("Core libraries loaded.")


Core libraries loaded.


In [2]:
# Q3(a): Hindi corpus containing 45 sentences.
hindi_sentences = ['मशीन लर्निंग डेटा से पैटर्न सीखती है', 'मशीन लर्निंग मॉडल भविष्यवाणी करते हैं', 'डीप लर्निंग न्यूरल नेटवर्क का उपयोग करती है', 'न्यूरल नेटवर्क जटिल पैटर्न सीखते हैं', 'कृत्रिम बुद्धिमत्ता जटिल समस्याओं को हल करती है', 'कृत्रिम बुद्धिमत्ता मशीन लर्निंग का उपयोग करती है', 'डेटा विज्ञान सांख्यिकी और प्रोग्रामिंग को जोड़ता है', 'डेटा विश्लेषण उपयोगी पैटर्न खोजने में मदद करता है', 'डेटा वैज्ञानिक बड़े डेटासेट पर मॉडल बनाते हैं', 'शोधकर्ता नए विचारों का परीक्षण करते हैं', 'कंप्यूटर विजन चित्रों का विश्लेषण करता है', 'कंप्यूटर विजन चित्रों में वस्तुओं का पता लगाता है', 'चित्र वर्गीकरण चित्रों को लेबल देता है', 'डीप लर्निंग चित्र वर्गीकरण को बेहतर बनाती है', 'प्राकृतिक भाषा प्रसंस्करण मानव भाषा का विश्लेषण करता है', 'भाषा मॉडल पाठ से पैटर्न सीखते हैं', 'प्राकृतिक भाषा मॉडल शब्दों और वाक्यों को संसाधित करते हैं', 'पाठ वर्गीकरण दस्तावेजों को श्रेणियों में बांटता है', 'छात्र कंप्यूटर विज्ञान में प्रोग्रामिंग सीखते हैं', 'छात्र एल्गोरिदम और डेटा संरचनाओं का अभ्यास करते हैं', 'प्रोग्रामिंग कौशल समस्याओं को हल करने में मदद करते हैं', 'कंप्यूटर विज्ञान के छात्र मशीन लर्निंग पढ़ते हैं', 'ऑनलाइन पाठ्यक्रम व्याख्यान और अभ्यास देते हैं', 'ऑनलाइन शिक्षा नए कौशल सीखने में मदद करती है', 'शिक्षा मंच छात्र की प्रगति का डेटा रखते हैं', 'शिक्षक सीखने के परिणामों को समझने के लिए डेटा का उपयोग करते हैं', 'एल्गोरिदम डेटा को उपयोगी परिणामों में बदलते हैं', 'एल्गोरिदम चित्रों और पाठ को वर्गीकृत कर सकते हैं', 'मॉडल प्रशिक्षण डेटा से सीखते हैं', 'प्रशिक्षण डेटा मशीन लर्निंग मॉडल को बेहतर बनाता है', 'बड़े डेटासेट न्यूरल नेटवर्क को सीखने में मदद करते हैं', 'शोधकर्ता बड़े डेटासेट पर मॉडल प्रशिक्षित करते हैं', 'पूर्वानुमान मॉडल भविष्य के परिणामों का अनुमान लगाते हैं', 'मशीन लर्निंग कई अनुप्रयोगों में पूर्वानुमान में मदद करती है', 'कृत्रिम बुद्धिमत्ता अनुसंधान नए मॉडल विकसित करता है', 'शोधकर्ता भाषा प्रसंस्करण के लिए एल्गोरिदम विकसित करते हैं', 'भाषा मॉडल संदर्भ के आधार पर शब्दों को दर्शाते हैं', 'समान संदर्भ वाले शब्दों के अर्थ संबंधित हो सकते हैं', 'संदर्भ शब्दों के अर्थ को समझने में मदद करता है', 'वितरणात्मक प्रतिनिधित्व संदर्भ के माध्यम से शब्दों का वर्णन करता है', 'डेटा मॉडल बुद्धिमान अनुप्रयोगों का समर्थन करते हैं', 'मशीन लर्निंग कंप्यूटर विजन में उपयोगी है', 'न्यूरल नेटवर्क भाषा प्रसंस्करण में उपयोगी हैं', 'शोधकर्ता डेटा का उपयोग करके मॉडल की तुलना करते हैं', 'छात्र कृत्रिम बुद्धिमत्ता और मशीन लर्निंग सीखते हैं']
assert 30 <= len(hindi_sentences) <= 50
print("Number of Hindi sentences:", len(hindi_sentences))


Number of Hindi sentences: 45


In [3]:
# Q3(b): Unicode normalization and tokenization.
hindi_normalized = [normalize_unicode(s) for s in hindi_sentences]
hindi_tokenized = [hindi_tokens(s) for s in hindi_normalized]

print("Original:", hindi_sentences[0])
print("Normalized:", hindi_normalized[0])
print("Tokens:", hindi_tokenized[0])


Original: मशीन लर्निंग डेटा से पैटर्न सीखती है
Normalized: मशीन लर्निंग डेटा से पैटर्न सीखती है
Tokens: ['मशीन', 'लर्निंग', 'डेटा', 'से', 'पैटर्न', 'सीखती', 'है']


In [4]:
# Q3(c): Train a Hindi Word2Vec model.
hindi_model = Word2Vec(
    sentences=hindi_tokenized,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=0,
    epochs=100,
    seed=42
)

print("Architecture: CBOW")
print("Vocabulary size:", len(hindi_model.wv))
print("Vector dimension:", hindi_model.vector_size)


Architecture: CBOW
Vocabulary size: 120
Vector dimension: 50


In [5]:
# Q3(d): Select 10 frequent Hindi words and display dense vectors.
frequency = pd.Series(
    [word for sentence in hindi_tokenized for word in sentence]
).value_counts()

selected_hindi_words = frequency.head(10).index.tolist()
print("Selected frequent words:", selected_hindi_words)

hindi_word_vectors = pd.DataFrame(
    [hindi_model.wv[w] for w in selected_hindi_words],
    index=selected_hindi_words
)

display(hindi_word_vectors.round(4))


Selected frequent words: ['हैं', 'है', 'को', 'का', 'में', 'डेटा', 'मॉडल', 'करते', 'लर्निंग', 'के']


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
हैं,0.0181,-0.1499,-0.2746,0.1854,-0.1684,0.0575,-0.1393,0.4269,-0.3416,-0.0306,...,0.0158,-0.1142,0.4169,-0.2337,0.1621,-0.3184,0.2216,0.5474,0.4331,-0.3784
है,0.0399,-0.1509,-0.2406,0.1606,-0.1457,0.0197,-0.1031,0.3444,-0.2737,-0.0047,...,0.0237,-0.0951,0.3650,-0.1976,0.1738,-0.3034,0.1823,0.4678,0.3736,-0.3493
को,0.0522,-0.2086,-0.3158,0.2000,-0.1876,0.0672,-0.1322,0.5068,-0.3955,-0.0068,...,0.0557,-0.1607,0.4797,-0.2734,0.2408,-0.3921,0.2456,0.6247,0.5215,-0.4824
का,0.0327,-0.1579,-0.3102,0.2209,-0.1896,0.0377,-0.1281,0.4852,-0.3801,-0.0027,...,0.0187,-0.1371,0.4412,-0.2453,0.2086,-0.4004,0.2612,0.5920,0.4910,-0.4539
में,0.0512,-0.1748,-0.2641,0.1841,-0.1767,0.0547,-0.1418,0.4312,-0.3372,-0.0040,...,0.0401,-0.1244,0.4285,-0.2415,0.2075,-0.3690,0.2238,0.5284,0.4309,-0.3808
डेटा,0.0264,-0.1308,-0.2788,0.1927,-0.1511,0.0219,-0.1255,0.3858,-0.3093,-0.0299,...,0.0325,-0.1261,0.3844,-0.2063,0.1827,-0.3227,0.2285,0.5078,0.3753,-0.3722
मॉडल,0.0175,-0.1496,-0.2774,0.1731,-0.1496,0.0258,-0.1092,0.3768,-0.3269,-0.0095,...,0.0427,-0.1251,0.4092,-0.2232,0.1723,-0.3227,0.2263,0.5027,0.3947,-0.3899
करते,0.0269,-0.1341,-0.2109,0.1336,-0.1008,0.0512,-0.1049,0.3178,-0.2571,-0.0006,...,0.0331,-0.1059,0.2934,-0.1589,0.1415,-0.2322,0.1439,0.3724,0.3001,-0.2896
लर्निंग,0.0435,-0.0962,-0.1970,0.1529,-0.1156,0.0302,-0.0899,0.2668,-0.2063,-0.0325,...,0.0399,-0.0961,0.2621,-0.1554,0.1252,-0.2250,0.1459,0.3836,0.2878,-0.2759
के,0.0563,-0.1598,-0.3040,0.1913,-0.1845,0.0560,-0.1485,0.4730,-0.3628,-0.0277,...,0.0521,-0.1225,0.4699,-0.2499,0.1943,-0.3877,0.2290,0.5907,0.4969,-0.4386


In [6]:
# Q3(e): Similarity between at least five Hindi word pairs.
hindi_pairs = [
    ("मशीन", "लर्निंग"),
    ("डेटा", "मॉडल"),
    ("कृत्रिम", "बुद्धिमत्ता"),
    ("भाषा", "प्रसंस्करण"),
    ("छात्र", "प्रोग्रामिंग")
]

word_results = []
for a, b in hindi_pairs:
    if a in hindi_model.wv and b in hindi_model.wv:
        word_results.append([a, b, hindi_model.wv.similarity(a, b)])

display(pd.DataFrame(
    word_results,
    columns=["शब्द 1", "शब्द 2", "Cosine Similarity"]
).round(4))


,शब्द 1,शब्द 2,Cosine Similarity
0,मशीन,लर्निंग,0.9954
1,डेटा,मॉडल,0.9985
2,कृत्रिम,बुद्धिमत्ता,0.9943
3,भाषा,प्रसंस्करण,0.9966
4,छात्र,प्रोग्रामिंग,0.9944


In [7]:
# Q3(f): Sentence representations = average of word embeddings.
hindi_sentence_vectors = np.vstack([
    average_vectors(tokens, hindi_model)
    for tokens in hindi_tokenized
])

print("Sentence-vector matrix shape:", hindi_sentence_vectors.shape)
print("Each sentence vector dimension:", hindi_model.vector_size)

display(pd.DataFrame(
    hindi_sentence_vectors[:5],
    index=[f"S{i}" for i in range(1, 6)]
).round(4))


Sentence-vector matrix shape: (45, 50)
Each sentence vector dimension: 50


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
S1,0.0212,-0.0968,-0.1849,0.1315,-0.1079,0.0206,-0.0779,0.2656,-0.2049,-0.0102,...,0.0223,-0.0873,0.2613,-0.1502,0.1218,-0.2282,0.1489,0.3588,0.2765,-0.2608
S2,0.0222,-0.1103,-0.1969,0.1381,-0.1148,0.0304,-0.0904,0.2876,-0.2320,-0.0161,...,0.0255,-0.0938,0.2798,-0.1604,0.1250,-0.2338,0.1504,0.3764,0.2951,-0.2732
S3,0.0232,-0.1005,-0.1744,0.1187,-0.1074,0.0201,-0.0726,0.2596,-0.2066,-0.0136,...,0.0163,-0.0784,0.2549,-0.1403,0.1135,-0.2148,0.1431,0.3364,0.2662,-0.2493
S4,0.0166,-0.1011,-0.1667,0.1055,-0.0989,0.0228,-0.0647,0.2390,-0.1883,-0.0116,...,0.0127,-0.0690,0.2329,-0.1325,0.0926,-0.1958,0.1324,0.3093,0.2484,-0.2248
S5,0.0259,-0.1191,-0.1857,0.1250,-0.1064,0.0286,-0.0759,0.2731,-0.2118,-0.0050,...,0.0165,-0.0820,0.2686,-0.1581,0.1319,-0.2217,0.1408,0.3569,0.2915,-0.2702


In [8]:
# Q3(g): Similarity between at least five pairs of Hindi sentences.
sentence_pairs = [(1, 2), (3, 4), (5, 6), (15, 16), (19, 20)]

sentence_results = []
for a, b in sentence_pairs:
    score = cosine_similarity(
        hindi_sentence_vectors[a-1].reshape(1, -1),
        hindi_sentence_vectors[b-1].reshape(1, -1)
    )[0, 0]
    sentence_results.append([f"S{a} - S{b}", score])

display(pd.DataFrame(
    sentence_results,
    columns=["Sentence Pair", "Cosine Similarity"]
).round(4))


,Sentence Pair,Cosine Similarity
0,S1 - S2,0.9995
1,S3 - S4,0.9996
2,S5 - S6,0.9996
3,S15 - S16,0.9994
4,S19 - S20,0.9996


### Q4(h): Word-level vs sentence-level representation

- A **word vector** represents one word as a dense fixed-length vector.
- A **sentence vector** in this notebook is created by averaging the vectors of its words.
- Both have the same dimensionality here (50), but they represent different linguistic units.
- Averaging is simple and useful for demonstrations, but it can lose word order and fine-grained meaning.


### Q3(i): Challenges in regional-language embeddings

1. **Morphology:** one root can have many inflected forms.
2. **Spelling variation:** different spellings can fragment the vocabulary.
3. **Unicode representation:** normalization is important for consistent text processing.
4. **Limited-resource settings:** smaller corpora provide less training evidence.
5. **Vocabulary coverage:** rare words may have weak or unreliable vectors.
6. **Contextual ambiguity:** the same word can have different meanings in different contexts.
